# 🎧 → 📄  Audiobook to Text (Google Colab)

Transcribe an audiobook (`.m4b`, `.mp3`, `.m4a`, `.wav`) to a text file — on Google's
free cloud computer with a **GPU**, so a 5-hour book takes **minutes**, and your own PC
does none of the heavy lifting.

**Just run each cell in order** (click the ▶ button on the left of a cell, or press
`Shift`+`Enter`). Read the short note above each one.

> **Two honest notes.** (1) Your audio is uploaded to Google's servers to do this — fine
> for a book you own, but it's not fully private like the desktop version. (2) Keep this
> browser tab open until it finishes (only a few minutes on the GPU); if you close it, the
> cloud computer disconnects.

## Step 0 — Turn on the free GPU  *(do this first)*

In the menu: **Runtime → Change runtime type → Hardware accelerator → `T4 GPU` → Save**.

Then run the cell below to confirm it's on.

In [ ]:
import subprocess
try:
    print(subprocess.check_output(['nvidia-smi', '-L']).decode().strip())
    print('\n✅ GPU is ON — transcription will be fast.')
except Exception:
    print('⚠️ No GPU yet. Do: Runtime → Change runtime type → T4 GPU → Save, then run this cell again.')

## Step 1 — Install the transcriber

Takes ~30 seconds. (Colab already has FFmpeg, so nothing else is needed.)

In [ ]:
!pip -q install faster-whisper
print('Installed. FFmpeg is already available on Colab.')

## Step 2 — Get your audiobook into Colab  *(pick ONE option)*

**Option A — Upload directly** (simplest; best for smaller files). Run the next cell,
click **Choose Files**, and pick your `.m4b`.

**Option B — From Google Drive** (best for big files, and it won't re-upload if you run
again). Skip Option A and use the Drive cell below instead.

In [ ]:
# OPTION A — upload straight from your computer.
# A big audiobook can be slow here; if it stalls, use Option B (Google Drive) below.
from google.colab import files
up = files.upload()
AUDIO_PATH = next(iter(up))          # the file you just picked
print('Using:', AUDIO_PATH)

In [ ]:
# OPTION B — from Google Drive (run this INSTEAD of Option A).
# 1) First put your .m4b in your Google Drive (drive.google.com), e.g. a folder 'Audiobooks'.
# 2) Run this cell and allow access when asked.
# 3) Edit the path below to point at your file, then run again.
from google.colab import drive
drive.mount('/content/drive')
AUDIO_PATH = '/content/drive/MyDrive/Audiobooks/YOUR_FILE.m4b'   # <-- change this line
import os
print('Found file:' if os.path.isfile(AUDIO_PATH) else 'NOT found — fix the path:', AUDIO_PATH)

## Step 3 — Transcribe

`MODEL` sets quality vs. speed. On the GPU you can afford a bigger model:
`tiny` (fastest) → `base` → `small` → `medium` → `large-v3` (best). **`small`** is a great
start; try **`medium`** for nicer wording. Then run the cell and watch the % climb.

In [ ]:
MODEL = 'small'      # tiny | base | small | medium | large-v3
LANGUAGE = None      # None = auto-detect; or force e.g. 'en'

import json, subprocess, textwrap, os
from faster_whisper import WhisperModel
import ctranslate2

try:
    AUDIO_PATH
except NameError:
    raise SystemExit('Set your file first: run the Option A upload cell OR the Option B Drive cell.')

# Read chapters (if the file has them) so the text gets chapter headings, like the desktop tool.
def get_chapters(path):
    try:
        out = subprocess.check_output(['ffprobe','-v','quiet','-print_format','json','-show_chapters', path]).decode()
        chs = json.loads(out).get('chapters', [])
        res = []
        for i, c in enumerate(chs):
            res.append((float(c.get('start_time', 0)), float(c.get('end_time', 0)),
                        (c.get('tags') or {}).get('title') or f'Chapter {i+1}'))
        return res
    except Exception:
        return []

chapters = get_chapters(AUDIO_PATH)
print(f'Found {len(chapters)} chapter(s).')

DEVICE = 'cuda' if ctranslate2.get_cuda_device_count() > 0 else 'cpu'
COMPUTE = 'float16' if DEVICE == 'cuda' else 'int8'
print('Loading model', MODEL, 'on', DEVICE, '— first time downloads it (~seconds)…')
model = WhisperModel(MODEL, device=DEVICE, compute_type=COMPUTE)

segments, info = model.transcribe(AUDIO_PATH, language=LANGUAGE, vad_filter=True)
total = float(getattr(info, 'duration', 0) or 0)
print(f'Language: {info.language} | length: {total/3600:.2f} h | transcribing…')

segs, last = [], 0.0
for s in segments:
    segs.append((s.start, s.end, s.text.strip()))
    if total and s.end - last >= 60:
        last = s.end
        print(f'  {100*s.end/total:5.1f}%', end='\r')
print('  100.0%  — done transcribing.        ')

def wrap(t): return '\n'.join(textwrap.wrap(' '.join(t.split()), 100))
OUT_NAME = os.path.splitext(os.path.basename(AUDIO_PATH))[0] + '.txt'
with open(OUT_NAME, 'w', encoding='utf-8') as f:
    if chapters:
        for cs, ce, title in chapters:
            body = ' '.join(t for (st, en, t) in segs if cs <= st < ce).strip()
            if body:
                f.write(f'\n## {title}\n\n' + wrap(body) + '\n\n')
    else:
        f.write(wrap(' '.join(t for _, _, t in segs)) + '\n')
print('Saved:', OUT_NAME)

## Step 4 — Download your text file

Run this to save the `.txt` to your computer's **Downloads** folder. (If you used Google
Drive, the commented line also drops a copy into your Drive.)

In [ ]:
from google.colab import files
files.download(OUT_NAME)

# Google Drive users can also keep a copy in Drive:
# import shutil; shutil.copy(OUT_NAME, '/content/drive/MyDrive/'); print('Copied to your Drive.')

---
### Tips & troubleshooting
- **How long?** On the T4 GPU, expect roughly a few minutes for a 5-hour book with `small`.
- **Nothing downloaded?** The file is written only when Step 3 prints `Saved:` — run Step 4 after that.
- **`No GPU` / very slow?** You skipped Step 0 — set the runtime to `T4 GPU` and run Steps 1–3 again.
- **Upload keeps failing?** Use Option B (Google Drive) instead of Option A.
- **Privacy:** for a fully-offline, nothing-leaves-your-PC run, use the desktop `aax2text` tool instead.